# BirdClef 2026 - Exploratory Data Analysis

Competition: Acoustic Species Identification in the Pantanal, South America

In [ ]:
import pandas as pd
import numpy as np
import os
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
DATA_DIR = Path('data/birdclef-2026')

## Load Data

In [ ]:
# Load CSVs
train_df = pd.read_csv(DATA_DIR / 'train.csv')
taxonomy_df = pd.read_csv(DATA_DIR / 'taxonomy.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
train_labels = pd.read_csv(DATA_DIR / 'train_soundscapes_labels.csv')

print(f"Train samples: {len(train_df)}")
print(f"Taxonomy entries: {len(taxonomy_df)}")
print(f"Sample submission rows: {len(sample_sub)}")
print(f"Train soundscape labels: {len(train_labels)}")

## Train Data Overview

In [ ]:
print("Train columns:", train_df.columns.tolist())
print("\nTrain head:")
train_df.head()

## Species Distribution

In [ ]:
# Count samples per species
species_counts = train_df['primary_label'].value_counts()
print(f"Number of unique species: {len(species_counts)}")
print(f"\nTop 10 species by count:")
print(species_counts.head(10))
print(f"\nBottom 10 species by count:")
print(species_counts.tail(10))

In [ ]:
# Plot species distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of sample counts
axes[0].hist(species_counts.values, bins=50, edgecolor='black')
axes[0].set_xlabel('Number of samples per species')
axes[0].set_ylabel('Number of species')
axes[0].set_title('Distribution of samples per species')

# Top 20 species
species_counts.head(20).plot(kind='barh', ax=axes[1])
axes[1].set_xlabel('Number of samples')
axes[1].set_title('Top 20 species by sample count')

plt.tight_layout()
plt.savefig('notebooks/species_distribution.png', dpi=150)
plt.show()

## Class Types

In [ ]:
# Class distribution (Aves, Amphibia, etc.)
class_counts = taxonomy_df['class_name'].value_counts()
print("Class distribution in taxonomy:")
print(class_counts)

In [ ]:
# Plot class distribution
fig, ax = plt.subplots(figsize=(10, 5))
class_counts.plot(kind='bar', ax=ax)
ax.set_xlabel('Class')
ax.set_ylabel('Number of species')
ax.set_title('Species by taxonomic class')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('notebooks/class_distribution.png', dpi=150)
plt.show()

## Audio Duration Analysis

In [ ]:
# Analyze audio file durations (sample from train_audio)
train_audio_dir = DATA_DIR / 'train_audio'
audio_files = list(train_audio_dir.glob('*/*.ogg'))[:100]  # Sample 100 files

durations = []
for f in audio_files:
    try:
        y, sr = librosa.load(str(f), sr=None)
        durations.append(len(y) / sr)
    except Exception as e:
        print(f"Error loading {f}: {e}")

print(f"Analyzed {len(durations)} audio files")
print(f"Mean duration: {np.mean(durations):.2f}s")
print(f"Median duration: {np.median(durations):.2f}s")
print(f"Min duration: {np.min(durations):.2f}s")
print(f"Max duration: {np.max(durations):.2f}s")

In [ ]:
# Plot duration distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(durations, bins=30, edgecolor='black')
ax.set_xlabel('Duration (seconds)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of audio file durations')
plt.tight_layout()
plt.savefig('notebooks/audio_duration.png', dpi=150)
plt.show()

## Train Soundscapes - Multi-label Analysis

In [ ]:
# Analyze train soundscape labels
train_labels['labels_list'] = train_labels['primary_label'].str.split(';')
train_labels['num_labels'] = train_labels['labels_list'].apply(len)

print(f"Total segments: {len(train_labels)}")
print(f"\nLabels per segment:")
print(train_labels['num_labels'].describe())

# Get all unique labels
all_labels = set()
for labels in train_labels['labels_list']:
    all_labels.update(labels)
print(f"\nUnique species in soundscapes: {len(all_labels)}")

In [ ]:
# Plot labels per segment
fig, ax = plt.subplots(figsize=(8, 5))
train_labels['num_labels'].value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_xlabel('Number of labels per segment')
ax.set_ylabel('Number of segments')
ax.set_title('Distribution of labels per 5-second segment')
plt.tight_layout()
plt.savefig('notebooks/labels_per_segment.png', dpi=150)
plt.show()

## Sample Submission Format

In [ ]:
# Understand submission format
print("Sample submission shape:", sample_sub.shape)
print("\nColumns (first 10):", sample_sub.columns[:10].tolist())
print("\nRow ID example:", sample_sub['row_id'].iloc[0])
print("\nFirst few rows:")
sample_sub.head(3)

## Geographic Distribution

In [ ]:
# Plot geographic distribution
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(train_df['longitude'], train_df['latitude'], 
                     alpha=0.3, s=5, c='blue')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Recording Locations')
plt.tight_layout()
plt.savefig('notebooks/geo_distribution.png', dpi=150)
plt.show()

print(f"Latitude range: {train_df['latitude'].min():.2f} to {train_df['latitude'].max():.2f}")
print(f"Longitude range: {train_df['longitude'].min():.2f} to {train_df['longitude'].max():.2f}")

## Data Summary

In [ ]:
# Summary statistics
print("=" * 50)
print("BirdClef 2026 - Data Summary")
print("=" * 50)
print(f"\nTraining data:")
print(f"  - Short audio clips: {len(train_df):,}")
print(f"  - Unique species: {train_df['primary_label'].nunique()}")
print(f"  - Average samples per species: {len(train_df) / train_df['primary_label'].nunique():.1f}")

print(f"\nTrain soundscapes:")
print(f"  - Total segments: {len(train_labels):,}")
print(f"  - Unique soundscapes: {train_labels['filename'].nunique()}")

print(f"\nTest data:")
print(f"  - Submission rows: {len(sample_sub):,}")
print(f"  - Target classes: {sample_sub.shape[1] - 1}")

print(f"\nTaxonomy:")
print(f"  - Total species: {len(taxonomy_df)}")
print(f"  - Classes: {taxonomy_df['class_name'].unique().tolist()}")

## Next Steps

- Build mel spectrogram generator
- Implement data augmentation
- Set up multi-label classification model
- Consider using pre-trained audio models (Perch, YAMNet)